# Sesi 11 — RAG bagian 1: Document Ingestion

**Yang akan Anda bangun di notebook ini**

Tujuh dokumen kebijakan LindungiKu masuk sebagai berkas `.md`, dan keluar sebagai
satu index FAISS yang siap dicari. Urutannya:

`dokumen` → `dibersihkan` → `dipecah` → `embedding` → `disimpan`

Di akhir notebook ada dua berkas baru di folder Anda, yaitu `index_lindungiku.faiss`
dan `potongan.json`. Notebook berikutnya memuat kedua berkas itu, jadi Anda tidak
perlu mengulang proses embedding.

## Intro to LangChain ?

Hari ini kita memakai LangChain untuk dua hal saja, yaitu memecah teks dan
menggabungkan dua jenis pencarian. Itu sebagian kecil dari kemampuannya. Supaya
Anda tahu apa yang ada di luar sana ketika membaca dokumentasinya sendiri:

| Bagian | Gunanya |
|---|---|
| **Document loaders** | Membaca PDF, HTML, Notion, Google Drive, database, dan puluhan sumber lain menjadi satu bentuk `Document` yang sama |
| **Text splitters** | Memecah dokumen panjang. **Kita pakai ini hari ini.** |
| **Embeddings + vector stores** | Pembungkus seragam untuk OpenAI, Gemini, Cohere, FAISS, Chroma, Pinecone, dan lainnya |
| **Retrievers** | Antarmuka pencarian yang seragam, termasuk hybrid search. **Kita pakai ini di notebook 2.** |
| **Chains dan LCEL** | Merangkai langkah menjadi satu alur yang bisa dijalankan, di-stream, dan dijalankan paralel |
| **Agents dan tools** | Membiarkan model memutuskan sendiri fungsi mana yang dipanggil |
| **Memory** | Menyimpan riwayat percakapan antar giliran |
| **Output parsers** | Memaksa keluaran model menjadi JSON atau objek Python yang tervalidasi |

**Mengapa hari ini kita hampir tidak memakai semuanya?** Karena setiap lapis
abstraksi yang Anda tambahkan adalah satu hal lagi yang harus ditelusuri ketika
jawabannya salah. Pada hari pertama membangun RAG, yang perlu Anda lihat adalah
pipeline-nya, bukan framework-nya. Setelah Anda pernah menulis setiap langkah
sendiri, LangChain menjadi penghemat waktu

In [ ]:
# perlu install langchain text splitters
# pip isntall langchain-text-splitters

In [ ]:
import json
import os
import pathlib
import re

import faiss
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

MODEL_EMBED = "gemini-embedding-001"
DIMENSI = 768


def akar_sesi():
    # Folder yang memuat kb_lindungiku, entah folder ini atau folder induknya.
    for p in (pathlib.Path("."), pathlib.Path("..")):
        if (p / "kb_lindungiku").is_dir():
            return p.resolve()
    raise SystemExit(
        "Folder kb_lindungiku tidak ditemukan. Jalankan notebook ini dari folder "
        "Session 11 atau dari salah satu subfoldernya."
    )


AKAR = akar_sesi()
FOLDER_KB = AKAR / "kb_lindungiku"
BERKAS_INDEX = AKAR / "index_lindungiku.faiss"
BERKAS_POTONGAN = AKAR / "potongan.json"

kunci = os.getenv("GEMINI_API_KEY")
if not kunci:
    raise SystemExit("GEMINI_API_KEY tidak ditemukan. Periksa file .env, lalu baca Troubleshooting bagian A.")

client = genai.Client(api_key=kunci)
print("akar sesi :", AKAR)
print("siap. dimensi embedding:", DIMENSI)

## Langkah 1 — Memuat dokumen

Tujuh dokumen kita akan diingest ke dalam bentuk teks. Hari ini kita mulai dari teks
bersih supaya fokusnya tetap pada RAG. Pada aplikasi nyata, teks bisa saja berupa pdf,
email, file excel, image, maupun dokumen lain

In [ ]:
def muat_dokumen(folder):
    dokumen = {}
    # TODO 1: baca setiap berkas .md di dalam folder.
    # Petunjuk: folder.glob("*.md") memberi daftar berkas,
    # dan berkas.read_text(encoding="utf-8") membaca isinya.
    # Simpan ke dalam dict dengan berkas.name sebagai kunci.
    return dokumen


dokumen = muat_dokumen(FOLDER_KB)

for nama, isi in dokumen.items():
    print(f"{nama:<38} {len(isi):>5} karakter  {len(isi.split()):>4} kata")
print(f"\ntotal {len(dokumen)} dokumen, {sum(len(t.split()) for t in dokumen.values())} kata")

## Langkah 2 — Data Cleaning

Header yang berulang, footer, nomor halaman, dan karakter sisa konversi PDF akan
ikut masuk ke dalam chunk. Setelah itu semuanya ikut masuk ke dalam vector, dan
mulai mencemari hasil pencarian.

Dokumen kita sudah cukup rapi, jadi fungsi ini pendek. Pada dokumen nyata, bagian
inilah yang paling panjang.

In [ ]:
def bersihkan(teks):
    teks = teks.replace("\u00a0", " ")          # non-breaking space
    teks = re.sub(r"[ \t]+", " ", teks)          # spasi berulang
    teks = re.sub(r"\n{3,}", "\n\n", teks)       # baris kosong berlebih
    teks = re.sub(r"^\s*Halaman \d+.*$", "", teks, flags=re.M)   # footer
    return teks.strip()


dokumen = {nama: bersihkan(isi) for nama, isi in dokumen.items()}
print("selesai dibersihkan.")
print(dokumen["01_kebijakan_klaim_kesehatan.md"][:280])

## Langkah 3 — Chunking

Kenapa kita perlu melakukan chunking ?

Model embedding dan LLM sama-sama membatasi panjang teks yang dapat diproses
sekaligus. Tetapi bukan itu alasan utamanya. Alasan utamanya adalah presisi. Satu
vector untuk satu dokumen utuh merata-ratakan seluruh isinya menjadi satu titik
yang kabur, dan tidak ada satu topik pun yang menonjol.

`RecursiveCharacterTextSplitter` mencoba daftar pemisah secara berurutan. Ia
memotong pada batas judul dahulu, lalu paragraf, lalu baris, lalu kalimat, dan
baru pada karakter apa pun sebagai pilihan terakhir. Artinya potongannya jatuh di
tempat yang masuk akal bagi manusia, bukan tepat di tengah kata.

`chunk_overlap` membuat setiap potongan membawa sedikit ekor potongan sebelumnya,
supaya kalimat yang terbelah di batas potongan tidak hilang maknanya.

In [ ]:
def pecah(teks, ukuran):
    # TODO 2: buat RecursiveCharacterTextSplitter, lalu kembalikan hasil split_text.
    # Pakai chunk_size=ukuran dan chunk_overlap=min(80, ukuran // 5).
    # Pakai separators=["\n## ", "\n\n", "\n", ". ", " ", ""] supaya potongan
    # jatuh di batas judul dan paragraf, bukan di tengah kata.
    pass


DOK = dokumen["01_kebijakan_klaim_kesehatan.md"]

for n in (6000, 80, 600):
    potongan_uji = pecah(DOK, n)
    rerata = sum(len(c) for c in potongan_uji) // len(potongan_uji)
    print(f"{n:>5}  {len(potongan_uji):>3} chunk  rata-rata {rerata:>4} karakter")

### Apa konsekuensi dari ukuran chunking yang tidak ideal ?

**Terlalu besar.** Satu chunk memuat seluruh dokumen. Ketika retrieval
mengembalikannya, model menerima sepuluh topik sekaligus padahal hanya satu yang
ditanyakan. Sisanya adalah noise.

**Terlalu kecil.** Perhatikan dokumen prosedur banding. Ada langkah yang merujuk
hasil langkah sebelumnya. Dengan chunk 80 karakter, langkah itu terputus dari
syarat yang memicunya, dan potongannya menjadi tidak bermakna sendirian.

In [ ]:
print("=" * 72)
print("UKURAN 6000 — satu chunk, sepuluh topik")
print("=" * 72)
besar = pecah(DOK, 6000)
print(f"jumlah chunk: {len(besar)}")
print("judul yang tercampur di dalam chunk pertama:")
for judul in re.findall(r"^## (.+)$", besar[0], flags=re.M):
    print("   -", judul)

print()
print("=" * 72)
print("UKURAN 80 — langkah terputus dari syaratnya")
print("=" * 72)
PROSEDUR = dokumen["04_prosedur_banding.md"]
kecil = pecah(PROSEDUR, 80)
print(f"jumlah chunk: {len(kecil)}")
for c in kecil[:6]:
    print(f"   [{c.strip()[:70]}]")

print()
print("=" * 72)
print("UKURAN 600 — satu chunk, satu gagasan utuh")
print("=" * 72)
pas = pecah(DOK, 600)
print(f"jumlah chunk: {len(pas)}")
print(pas[0][:400])

### Memotong seluruh knowledge base

Kita akan gunakan chunkg size 600. Setiap potongan disimpan bersama nama dokumen asalnya,
supaya nanti kita bisa menjawab pertanyaan "ini datang dari mana?" ketika hasil
retrieval terlihat aneh. Menyimpan asal-usul potongan adalah kebiasaan yang akan
menyelamatkan Anda saat debugging.

In [ ]:
UKURAN = 600

potongan = []
for nama, isi in dokumen.items():
    for teks in pecah(isi, UKURAN):
        potongan.append({"sumber": nama, "teks": teks})

print(f"{len(potongan)} chunk dari {len(dokumen)} dokumen")
for nama in dokumen:
    n = sum(1 for p in potongan if p["sumber"] == nama)
    print(f"   {nama:<38} {n:>3} chunk")

## Langkah 4 — Embedding

Sekarang setiap potongan teks diubah menjadi deretan angka. Dua hal yang wajib
Anda perhatikan di sini, dan keduanya sering terlewat.

**Pertama, `task_type`.** Gemini menghasilkan embedding yang berbeda untuk dokumen
dan untuk pertanyaan. Dokumen memakai `RETRIEVAL_DOCUMENT`, pertanyaan memakai
`RETRIEVAL_QUERY`. Model tahu bahwa keduanya akan dibandingkan satu sama lain, dan
menempatkannya di ruang yang sama dengan cara yang tepat. Memakai tipe yang salah
tidak memunculkan error, hanya membuat hasil pencarian menjadi lebih buruk tanpa
Anda sadari.

**Kedua, normalisasi.** `gemini-embedding-001` secara bawaan mengembalikan 3072
dimensi. Kita meminta 768 lewat `output_dimensionality`. Yang dikembalikan adalah
potongan awal dari vector 3072 dimensi tersebut, dan potongan itu **tidak lagi
memiliki panjang satu**. Kita harus menormalkannya sendiri.

Mengapa panjangnya harus satu? Karena FAISS tidak memiliki index cosine. Yang ada
adalah `IndexFlatIP`, yaitu inner product. Rumusnya:

```
a · b = |a| × |b| × cos(a, b)
```

Kalau `|a|` dan `|b|` sama-sama satu, maka inner product **sama persis dengan**
cosine similarity. Jadi kita tidak mengubah metriknya, kita menyesuaikan datanya
supaya cocok dengan metrik yang tersedia.

In [ ]:
def normalisasi(v):
    # TODO 3: bagi setiap baris dengan panjangnya sendiri, supaya panjangnya menjadi 1.
    # Petunjuk: np.linalg.norm(v, axis=1, keepdims=True) memberi panjang tiap baris.
    # Jangan lupa menjaga agar tidak terjadi pembagian dengan nol.
    pass


def embed(daftar_teks, task_type, batas=100):
    hasil = []
    for i in range(0, len(daftar_teks), batas):
        bagian = daftar_teks[i : i + batas]
        resp = client.models.embed_content(
            model=MODEL_EMBED,
            contents=bagian,
            # TODO 4: lengkapi config-nya.
            # Pakai types.EmbedContentConfig dengan task_type=task_type
            # dan output_dimensionality=DIMENSI.
        )
        hasil.extend(e.values for e in resp.embeddings)
    return np.array(hasil, dtype="float32")


mentah = embed([p["teks"] for p in potongan], "RETRIEVAL_DOCUMENT")
print("bentuk        :", mentah.shape)
print("panjang sebelum normalisasi:", np.linalg.norm(mentah[0]).round(4))

vec = normalisasi(mentah)
print("panjang sesudah normalisasi:", np.linalg.norm(vec[0]).round(4))

## Langkah 5 — Membangun index dan menyimpannya

`IndexFlatIP` menyimpan semua vector apa adanya dan membandingkan pertanyaan
dengan setiap vector satu per satu. Untuk 44 potongan itu seketika. Untuk jutaan
potongan Anda akan memakai index berjenis lain yang mencari secara aproksimasi,
lebih cepat tetapi tidak selalu tepat. Kata "Flat" berarti tanpa aproksimasi.

Kita menyimpan dua berkas. Index berisi vector-nya. `potongan.json` berisi teks
aslinya, karena FAISS hanya mengembalikan nomor baris, bukan teks.

In [ ]:
# TODO 5: buat IndexFlatIP dengan DIMENSI, masukkan vec ke dalamnya,
# lalu cetak index.ntotal untuk memastikan jumlahnya sesuai.
index = None

faiss.write_index(index, str(BERKAS_INDEX))
BERKAS_POTONGAN.write_text(
    json.dumps(potongan, ensure_ascii=False, indent=1), encoding="utf-8"
)
print("tersimpan:")
print("  ", BERKAS_INDEX)
print("  ", BERKAS_POTONGAN)